# 16 · QLoRA ViVQA 2 epoch — chọn epoch trên val (W9)

**Vấn đề:** B2 trên ViVQA (W07) mới train **1 epoch** (EM test 0,7378). Chưa biết train lâu hơn có tốt hơn không, và nếu chọn số epoch bằng cách nhìn **test** thì là rò rỉ (chọn mô hình trên chính tập báo cáo). Cần chọn trên **val**, rồi chỉ chạm test **một lần** cho mô hình được chọn.

**Vấn đề phụ (rò rỉ ảnh trong val):** val ViVQA được tách từ `train.csv` **theo câu hỏi**, nên nhiều câu val dùng chung ảnh với câu train. Train càng lâu, model càng "nhớ ảnh" → val dễ thiên vị epoch nhiều hơn. Vì vậy chọn epoch trên **val sạch** = các câu val có ảnh **không xuất hiện** trong `train.json`.

**Ứng viên** (chọn theo luật ghi trước trong `docs/W09/README.md`):

| Ứng viên | Mô hình | Step |
|---|---|---|
| C1 | B2 W07 (lịch 1 epoch) — đã có | 675 |
| C2 | Run mới lịch 2 epoch, adapter sau epoch 1 | 675 |
| C3 | Run mới lịch 2 epoch, adapter sau epoch 2 | 1.350 |

**Thứ tự:** eval C1 trên val sạch (≈ 6–12 phút, kiểm pipeline sớm) → train 2 epoch (≈ 6,1 giờ) → eval C2, C3 trên val sạch → chọn theo luật → eval test **chỉ khi** chọn C2/C3 → tự trả máy. Tổng ≈ 7–7,5 giờ L4 (≈ 11 CU). Chạy qua đêm được: bị ngắt thì Run all lại, phần đã xong bỏ qua, train chạy tiếp từ checkpoint (lưu mỗi 100 step ≈ 27 phút).

## 1. Thiết lập môi trường

Giống notebook 14/15 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (có notebook 16), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('notebooks/16_qlora_vivqa_2epoch_select_qwen25vl.ipynb'), 'Code cũ -> git pull chưa lấy được commit notebook 16'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B2 W07). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run, val sạch, kiểm tra trước khi tốn GPU

- Adapter B2 W07 (C1) trên Drive phải **đúng sha256** đã chấm ở W07 (đọc từ `run_signature.json` trên git).
- Tạo **val sạch** `data/vivqa/val_imgdisjoint.json` (giữ thứ tự gốc, không ngẫu nhiên): câu val có ảnh không nằm trong `train.json`. Nếu < 300 câu → dùng **toàn bộ val** (ghi rõ trong biên bản).
- Biên bản tách: `experiments/W09_vivqa_epoch_select_L4_seed42/val_split_manifest.json` (sha256, số câu, số ảnh).
- Selftest: `train_qlora`, `run_baseline`.

*Kỳ vọng: `adapter C1 sha256 khớp ✅`, dòng `val: 1200 câu | ảnh trùng train … | val sạch … câu`, dòng `-> chọn epoch trên: …`, `✅ selftest OK (2 script)`.*

In [ ]:
import json, hashlib, subprocess
import pandas as pd
from src.eval.run_baseline import adapter_fingerprint

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vivqa'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
ADP_ROOT = f'{DRIVE_ROOT}/adapters'
TRAIN_FILE = f'{DRIVE_ROOT}/data/vivqa/train.json'
VAL_FILE = f'{DRIVE_ROOT}/data/vivqa/val.json'
VAL_CLEAN = f'{DRIVE_ROOT}/data/vivqa/val_imgdisjoint.json'
OUT = f'{EXP_ROOT}/W09_vivqa_epoch_select_L4_seed42'
SAVE_STEPS = 100
MIN_CLEAN = 300
os.makedirs(OUT, exist_ok=True)

W07_TRAIN = 'W07_B2_qlora_r16_qwen25vl_vivqa_ep1_bf16_L4_seed42'
W07_TEST = 'W07_B2_eval_qlora_r16_ep1_vivqa_test_bf16_L4_seed42'
TRAIN2 = 'W09_B2_qlora_r16_qwen25vl_vivqa_ep2_bf16_L4_seed42'
ADAPTER = {'C1': f'{ADP_ROOT}/{W07_TRAIN}/epoch_1',
           'C2': f'{ADP_ROOT}/{TRAIN2}/epoch_1',
           'C3': f'{ADP_ROOT}/{TRAIN2}/epoch_2'}
STEPS = {'C1': 675, 'C2': 675, 'C3': 1350}
DESC = {'C1': 'B2 W07 (lịch 1 epoch)', 'C2': 'lịch 2 epoch · sau epoch 1', 'C3': 'lịch 2 epoch · sau epoch 2'}

want = json.load(open(f'experiments/{W07_TEST}/run_signature.json', encoding='utf-8'))['adapter']['weights_sha256']
got = adapter_fingerprint(ADAPTER['C1'])['weights_sha256']
assert got == want, f'adapter C1 trên Drive KHÁC adapter đã chấm W07 ({got[:12]} ≠ {want[:12]}) -> dừng, báo Claude'
print(f'adapter C1 sha256 khớp ✅ ({got[:12]})')

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

train = json.load(open(TRAIN_FILE, encoding='utf-8'))
val = json.load(open(VAL_FILE, encoding='utf-8'))
train_imgs = {r['image'] for r in train}
clean = [r for r in val if r['image'] not in train_imgs]
print(f"val: {len(val)} câu | ảnh trùng train {sum(r['image'] in train_imgs for r in val)} câu | val sạch {len(clean)} câu "
      f"/ {len({r['image'] for r in clean})} ảnh")
if not os.path.exists(VAL_CLEAN):
    json.dump(clean, open(VAL_CLEAN, 'w', encoding='utf-8'), ensure_ascii=False)
assert [r['question_id'] for r in json.load(open(VAL_CLEAN, encoding='utf-8'))] == [r['question_id'] for r in clean], \
    'val_imgdisjoint.json trên Drive khác bản tính lại -> dừng, báo Claude'
SEL_FILE, SEL_NAME = (VAL_CLEAN, 'valclean') if len(clean) >= MIN_CLEAN else (VAL_FILE, 'valfull')
print(f'-> chọn epoch trên: {SEL_NAME} ({SEL_FILE.split("/")[-1]})')
json.dump({'what': 'Tập chọn epoch ViVQA (W09): val câu có ảnh không nằm trong train.json; thứ tự giữ như val.json',
           'train_sha256': sha256_file(TRAIN_FILE), 'val_sha256': sha256_file(VAL_FILE),
           'val_clean_sha256': sha256_file(VAL_CLEAN), 'n_train': len(train), 'n_val': len(val),
           'n_val_clean': len(clean), 'n_val_clean_images': len({r['image'] for r in clean}),
           'min_clean_rule': MIN_CLEAN, 'selection_set': SEL_NAME},
          open(f'{OUT}/val_split_manifest.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

EVAL_SEL = {c: f'W09_{c}_eval_qlora_r16_vivqa_{SEL_NAME}_bf16_L4_seed42' for c in STEPS}
EVAL_TEST = {c: f'W09_{c}_eval_qlora_r16_vivqa_test_bf16_L4_seed42' for c in ('C2', 'C3')}

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên')

def eval_run(cand, data_file, run_name):
    if os.path.exists(f'{EXP_ROOT}/{run_name}/metrics.json'):
        print(f'⏭ {run_name}: đã có -> bỏ qua')
    else:
        run_cmd(f'python -u -m src.eval.run_baseline --dataset {DS} --data_file {data_file} --run_name {run_name} '
                f'--adapter {ADAPTER[cand]}')
    m = json.load(open(f'{EXP_ROOT}/{run_name}/metrics.json', encoding='utf-8'))['metrics']
    print(f"✔ {cand} ({DESC[cand]}): EM={m['exact_match']:.4f} ANLS={m['anls']:.4f}")

outs = [subprocess.run(f'python -m {m} --selftest', shell=True, capture_output=True, text=True).stdout
        for m in ('src.adaptation.train_qlora', 'src.eval.run_baseline')]
assert all('SELFTEST OK' in o for o in outs), outs
print('✅ selftest OK (2 script)')

## 3. Eval C1 (B2 W07) trên tập chọn epoch (≈ 6–12 phút)

Chạy **trước** khi train 6 giờ để chắc pipeline val chạy được.

*Kỳ vọng: `== KẾT QUẢ: EM=…`, dòng `✔ C1 …`.*

In [ ]:
eval_run('C1', SEL_FILE, EVAL_SEL['C1'])

## 4. Train QLoRA ViVQA lịch 2 epoch (≈ 6,1 giờ)

Cùng lệnh với B2 W07, chỉ đổi `--epochs 2`. Adapter cuối mỗi epoch lưu riêng ở `epoch_1/` và `epoch_2/`. Checkpoint mỗi 100 step (≈ 27 phút).

*Kỳ vọng: `lịch: 1350 step tổng (675 step/epoch)`, `💾 adapter epoch 1`, `💾 adapter epoch 2`, `== XONG: 1350 step`. Bị ngắt → Run all lại → `↻ CHẠY TIẾP từ checkpoint-…`.*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{TRAIN2}/train_log.json'):
    print('⏭ train 2 epoch: đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.adaptation.train_qlora --dataset {DS} --run_name {TRAIN2} --epochs 2 --save_steps {SAVE_STEPS}')
meta = json.load(open(f'{EXP_ROOT}/{TRAIN2}/train_log.json', encoding='utf-8'))['meta']
assert meta['seed'] == 42 and meta['training']['total_steps'] == 1350, (meta['seed'], meta['training']['total_steps'])
assert os.path.isdir(ADAPTER['C2']) and os.path.isdir(ADAPTER['C3']), 'Thiếu adapter epoch_1 / epoch_2'
print(f"✔ train: {meta['cost']['global_step']} step | {meta['cost']['sec_per_step']} s/step | VRAM {meta['cost']['peak_vram_gb']} GB | "
      f"chạy tiếp từ {meta['resumed_from']}")

## 5. Eval C2, C3 trên tập chọn epoch (≈ 12–25 phút)

*Kỳ vọng: 2 dòng `✔ C2 …`, `✔ C3 …`.*

In [ ]:
for c in ('C2', 'C3'):
    eval_run(c, SEL_FILE, EVAL_SEL[c])

## 6. Chọn epoch theo luật (ghi trước trong README W09)

1. Ứng viên có **EM cao nhất** trên tập chọn; hoà EM → ít step hơn → C1.
2. Nếu ứng viên đó ≠ C1 nhưng so với C1 **McNemar exact p ≥ 0,05** → **giữ C1** (rẻ hơn, nguyên tắc efficiency).
3. Chỉ ứng viên được chọn (nếu ≠ C1) mới được chấm trên **test** (mục 7).

*Kỳ vọng: bảng 3 ứng viên, 2 dòng so sánh với C1, dòng `➜ CHỌN: …`.*

In [ ]:
rows = []
for c in STEPS:
    m = json.load(open(f'{EXP_ROOT}/{EVAL_SEL[c]}/metrics.json', encoding='utf-8'))['metrics']
    rows.append({'ứng viên': c, 'mô tả': DESC[c], 'step': STEPS[c], 'EM': round(m['exact_match'], 4), 'ANLS': round(m['anls'], 4)})
display(pd.DataFrame(rows))

def paired(run_a, run_b, la, lb, out):
    if not os.path.exists(out):
        r = subprocess.run(f'python src/eval/compare_runs.py --run_a {EXP_ROOT}/{run_a} --run_b {EXP_ROOT}/{run_b} '
                           f'--label_a {la} --label_b {lb} --out {out}', shell=True, capture_output=True, text=True)
        assert r.returncode == 0, r.stdout + r.stderr
    p = json.load(open(out, encoding='utf-8')); p.pop('only_b_question_ids', None)
    print(f"{la} -> {lb}: EM {p['em_a']:.4f} -> {p['em_b']:.4f} | chỉ {la} đúng {p['n_only_a']}, chỉ {lb} đúng {p['n_only_b']}, "
          f"p={p['p_value']:.3g} | ΔANLS {p['anls_diff']:+.4f} CI95 [{p['anls_ci95'][0]:+.4f}; {p['anls_ci95'][1]:+.4f}]")
    return p

PAIRS = {f'C1_vs_{c}_{SEL_NAME}': paired(EVAL_SEL['C1'], EVAL_SEL[c], 'C1', c, f'{OUT}/paired_C1_vs_{c}_{SEL_NAME}.json')
         for c in ('C2', 'C3')}
em = {r['ứng viên']: r['EM'] for r in rows}
best = sorted(STEPS, key=lambda c: (-em[c], STEPS[c], c != 'C1'))[0]
if best == 'C1':
    CHOSEN, why = 'C1', 'C1 có EM cao nhất (hoặc hoà, ít step hơn)'
elif PAIRS[f'C1_vs_{best}_{SEL_NAME}']['p_value'] >= 0.05:
    CHOSEN, why = 'C1', f'{best} EM cao nhất nhưng không khác C1 có ý nghĩa (p ≥ 0,05) -> giữ C1 (rẻ hơn)'
else:
    CHOSEN, why = best, f'{best} EM cao nhất và hơn C1 có ý nghĩa (p < 0,05)'
print(f'\n➜ CHỌN: {CHOSEN} ({DESC[CHOSEN]}) — {why}')

## 7. Test (3.001 câu) — chỉ khi chọn C2/C3 (≈ 30 phút)

Chọn C1 → **không** chạm test lần nữa: B2 W07 (EM 0,7378) giữ nguyên làm B2 ViVQA chính thức.
Chọn C2/C3 → chấm test **một lần**, so ghép cặp với B2 W07 trên test: "CẢI THIỆN" nếu EM tăng **và** p < 0,05; còn lại báo đúng như kết quả.

*Kỳ vọng: `C1 được chọn -> không chạy test` hoặc `== KẾT QUẢ` + dòng `➜ TEST: …`; file `epoch_select_summary.json`.*

In [ ]:
TEST_RES, test_verdict = None, None
if CHOSEN == 'C1':
    print('C1 được chọn -> không chạy test (B2 W07 giữ nguyên, EM test 0,7378)')
else:
    eval_run(CHOSEN, f'{DRIVE_ROOT}/data/vivqa/test.json', EVAL_TEST[CHOSEN])
    TEST_RES = p = paired(W07_TEST, EVAL_TEST[CHOSEN], 'B2_W07', CHOSEN, f'{OUT}/paired_B2W07_vs_{CHOSEN}_test.json')
    test_verdict = ('CẢI THIỆN' if p['em_b'] > p['em_a'] and p['p_value'] < 0.05 else
                    'KÉM HƠN có ý nghĩa' if p['em_b'] < p['em_a'] and p['p_value'] < 0.05 else 'KHÔNG KHÁC BIỆT có ý nghĩa')
    print(f'➜ TEST: {CHOSEN} so với B2 W07: {test_verdict}')
json.dump({'what': 'W09: chọn số epoch QLoRA ViVQA trên val sạch (ảnh không trùng train), test chỉ cho ứng viên được chọn',
           'rule': 'docs/W09/README.md mục Luật chọn epoch ViVQA', 'selection_set': SEL_NAME,
           'val_split_manifest': json.load(open(f'{OUT}/val_split_manifest.json', encoding='utf-8')),
           'candidates': rows, 'paired_selection': PAIRS, 'chosen': CHOSEN, 'why': why,
           'train_2ep_cost': meta['cost'], 'train_2ep_resumed_from': meta['resumed_from'],
           'test_paired_vs_B2W07': TEST_RES, 'test_verdict': test_verdict, 'gpu': GPU},
          open(f'{OUT}/epoch_select_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('->', f'{OUT}/epoch_select_summary.json')

## 8. Tự trả máy L4 khi chạy xong

Kết quả đã nằm trên Drive. Output phía trên vẫn hiện trong tab sau khi trả máy. Notebook dừng sớm vì lỗi thì máy **không** tự trả → tự bấm **Runtime → Disconnect and delete runtime** sau khi đọc lỗi.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy L4 …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy L4 để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 9. ⏸ DỪNG

Gửi Claude output **mục 2, 6 và 7**. Nếu tab đã đóng mất output: không sao — mọi kết quả nằm trong `experiments/W09_vivqa_epoch_select_L4_seed42/` trên Drive, Cursor chép và kiểm lại được.